# DAPSTOM 6.4 Local Snapshot — Dataset Explanation, EDA and Analysis Plan

This notebook documents the exact Microsoft Access snapshot supplied as CEFAS/DAPSTOM data, summarises its analytical content and limitations, and defines a reproducible analysis plan. The companion `DAPSTOM_EDA_visuals.ipynb` contains distributions and representative individual networks.

## Executive Summary

- Relational chain: `PROVENANCE → HAUL → PREDATOR → PREY`, with taxonomy linked by TSN.
- Scale: **10,262 hauls**, **132,647 predator records**, **481,931 represented stomachs**, and **283,121 prey records**.
- Coverage: **1836–2023**, with uneven spatial, temporal and source effort.
- Primary resolved network rule: `prey_tsn > 0`, TSN node identity, directed relation `predator → consumed prey`, self-links retained but reported separately.
- Primary graph across all strata: **S=1,787 TSN nodes**, **L=10,163 links**, non-self density **0.0032**.
- The dataset supports food-web and link-prediction work, but comparisons must account for pooling, effort, provenance, taxonomic resolution and calculated rather than observed weights.

## 1. Scope, Provenance and Citation

The analysed object is the local file labelled `DAPSTOM 6-4 COMBINED`, size **96,264,192 bytes**, SHA-256 `5a1346cd28e98b73ec2a51819408ccf333b76b108a39d2c3f3d3a3554a6f290e`. The original supplied file and working copy are byte-identical (see validation table below), and extraction is read-only.

Custodian and official record: **Centre for Environment, Fisheries & Aquaculture Science (Cefas)**, [DOI 10.14466/CefasDataHub.144](https://doi.org/10.14466/CefasDataHub.144), listed under the Open Government Licence. The public description attributes the initiative to Defra/EU support and describes digitised fish-stomach records from logbooks/reports, partner contributions and publications.

**Version note:** the local file and schema are labelled **6.4**, while the available public description/report refers to **6.3**. The report also gives 481,476 stomachs, whereas the local snapshot sums to 481,931. Therefore this analysis cites the exact checksum and treats the 6.4-to-6.3 release lineage as a Cefas confirmation item before publication.

The `PROVENANCE` table contains **741 campaigns/sources** and is complete for its five inspected fields:

| Source type | Campaign/source rows | Linked hauls |
| --- | --- | --- |
| MAFF/CEFAS DATASET | 270 | 4,121 |
| PAPER | 251 | 1,900 |
| REPORT | 90 | 1,570 |
| MAFF/CEFAS REPORT | 72 | 668 |
| PARTNER | 41 | 1,850 |
| CHARTER | 10 | 141 |
| THESIS | 4 | 8 |
| COMMERCIAL | 3 | 4 |

The auxiliary `cefas_arctic_cruises_and_catches_1930_1959.csv` is **outside the scope** of this Access-database EDA and is not silently merged.

## 2. Relational Model, Variables, Units and Identifiers

Row meaning:

- `HAUL`: a sampling event/location.
- `PREDATOR`: an individual or pooled stomach record; one row is not necessarily one stomach.
- `PREY`: one prey item/category attached to a predator record; multiple rows may share `pred_id`.
- `PROVENANCE`: campaign/source documentation linked by `cruise_name`.

`PROVENANCE.cruise_name → HAUL.cruise_name →(haul_id) PREDATOR →(pred_id) PREY`; TSN links predator/prey records to taxonomy.

| Table | Variables | Meaning / key role | Unit or type | Analytical note |
| --- | --- | --- | --- | --- |
| PROVENANCE | `cruise_name` | Source/campaign identifier; FK from HAUL | identifier | Links every haul to source documentation. |
| PROVENANCE | `uploaded`, `source_type`, `data_input`, `data_derived_from` | Version introduced, source class, digitiser and source description | categorical/text | All 741 provenance rows are populated. |
| HAUL | `haul_id` | Sampling-event identifier; PK | identifier | Parent key for predator records. |
| HAUL | `Year`, `Month`, `Day`, `date` | Sampling date | calendar fields | Month/day/date are less complete than year. |
| HAUL | `shot_lat_dd`, `shot_lon_dd` | Sampling position | decimal degrees | Point coordinates are partial. |
| HAUL | `ices_rect`, `ices_division`, `sea` | Nested spatial descriptors | categorical | ICES rectangle is 0.5° latitude × 1° longitude. |
| HAUL | `shot_depth_m` | Sampling depth | m | Stored as text; parseability and values such as `INTERTIDAL` require handling. |
| PREDATOR | `pred_id`, `haul_id` | Predator-record PK and FK to HAUL | identifier | A record may represent one or pooled stomachs. |
| PREDATOR | `pred`, `tsn` | Local predator label and ITIS Taxonomic Serial Number | code/identifier | Use TSN, not the short label, as the graph node key. |
| PREDATOR | `pred_length_cm`, `mean_length_cm` | Observed or pooled mean predator length | cm | Important for ontogenetic diet effects. |
| PREDATOR | `pred_wgt_g` | Predator weight | g | Stored as text and incompletely populated. |
| PREDATOR | `pooled`, `num_stomachs`, `num_empty` | Sampling-unit flag and represented stomach quantities | flag/count-like | Some recorded quantities are fractional; do not force integer type blindly. |
| PREY | `id`, `pred_id` | Prey-record PK and FK to PREDATOR | identifier | Many prey rows can belong to one predator record. |
| PREY | `prey_name`, `tsn`, `qual_code` | Prey label, ITIS TSN and life-stage/sex qualifier | text/identifier | Non-positive TSN values are sentinel or unresolved categories. |
| PREY | `prey_length`, `ind_prey_wgt_g` | Prey length and individual prey weight | cm / g | Both are stored as text and need parsing rules. |
| PREY | `digestion` | Source-specific digestion-state index | index | Documentation includes an unknown code; harmonise before modelling. |
| PREY | `min_num` | Recorded minimum prey quantity | count-like | A minimum/derived quantity, not always an exact integer count. |
| PREY | `cpw` | Calculated prey weight | g | Usually derived from mean mass or length-weight relationships, not directly observed biomass. |
| TAXONOMY | `tsn`, `aphiaid`, hierarchy, functional groups | ITIS/WoRMS identifiers and taxonomic/functional attributes | identifier/categorical | Taxonomic resolution and AphiaID coverage are incomplete. |

## 3. Ecological and Analytical Meaning

An observed DAPSTOM edge is evidence that a prey category occurred in the realised diet of a sampled predator under a particular place, time, body size, campaign and recording protocol. The selected database direction is **predator → consumed prey**; ecological energy flow is the reverse. An unobserved edge is not evidence of ecological absence.

`min_num` is a recorded minimum quantity and may encode source-derived evidence; `cpw` approximates prey mass using calculation rules and should not be presented as uniformly observed biomass. Ontogeny, pooling, sampling effort and provenance can all alter apparent diet and network structure.

In [ ]:
from pathlib import Path
import csv, json

TABLE_DIR = Path('tables')
if not TABLE_DIR.exists():
    TABLE_DIR = Path('data/processed/dapstom_eda/tables')

def load_csv(name):
    with (TABLE_DIR / f'{name}.csv').open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

manifest = json.loads((TABLE_DIR / 'eda_manifest.json').read_text())
sorted(path.name for path in TABLE_DIR.glob('*.csv'))

## 4. Dataset Dimensions and Coverage

| Table | Rows |
| --- | --- |
| HAUL 6-4 COMBINED | 10,262 |
| ICES 6-4 COMBINED | 10,261 |
| MURRAY_TAXONOMY | 1,805 |
| PREDATOR 6-4 COMBINED | 132,647 |
| PRED_TAXONOMY 6-4 COMBINED | 210 |
| PREY 6-4 COMBINED | 283,121 |
| PREY_TAXONOMY 6-4 COMBINED | 1,718 |
| PROVENANCE 6-4 COMBINED | 741 |
| QUALIFYER 6-4 | 21 |
| SHIPS 6-4 COMBINED | 93 |

Spatial coverage by available resolution:

| Resolution | Hauls | Coverage |
| --- | --- | --- |
| lat_lon | 6,799 | 66.3% |
| ices_rectangle | 8,698 | 84.8% |
| ices_division | 10,262 | 100.0% |
| sea | 10,262 | 100.0% |

Largest sea-level groups:

| Sea | Hauls | With lat/lon | With ICES rectangle | Distinct rectangles |
| --- | --- | --- | --- | --- |
| North Sea | 3,665 | 67.8% | 97.1% | 182 |
| Celtic Sea | 1,513 | 30.9% | 31.3% | 57 |
| Irish Sea | 1,054 | 75.8% | 83.1% | 26 |
| Channel | 1,006 | 30.5% | 98.4% | 24 |
| Spitzbergen (Greenland Sea) | 924 | 97.9% | 97.9% | 94 |
| Freshwater | 426 | 94.8% | 94.8% | 40 |
| W Scotland | 333 | 74.8% | 85.0% | 39 |
| Norwegian Sea | 276 | 94.9% | 99.6% | 171 |
| Faeroes | 196 | 93.9% | 100.0% | 21 |
| Kattegat | 160 | 100.0% | 100.0% | 10 |

## 5. Sampling Unit: Individual and Pooled Stomachs

| Pooled | Predator rows | % rows | Represented stomachs | % stomachs | Mean stomachs/row |
| --- | --- | --- | --- | --- | --- |
| n | 122,764 | 92.5% | 122,864 | 25.5% | 1.00 |
| y | 9,879 | 7.4% | 358,838 | 74.5% | 36.32 |
| (missing) | 4 | 0.0% | 229 | 0.0% | 57.25 |

Pooled rows are a minority of database rows but dominate represented stomachs. Network comparisons must therefore report both row counts and effort expressed as hauls/stomachs.

## 6. Descriptive Statistics, Distributions and Outliers

Tukey fences (`Q1 - 1.5×IQR`, `Q3 + 1.5×IQR`) are used as **review flags**, not automatic deletion rules. Heavy tails are expected for pooled effort and dietary quantities. The companion visual notebook shows the corresponding continuous-variable histograms (log-transformed where appropriate).

| Variable | Unit | Non-null n | Missing | Median [Q1, Q3] | Mean ± SD | P99 | Max | Above Tukey fence |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Predator length | cm | 122,643 | 10,004 | 30.00 [20.80, 45.00] | 35.15 ± 21.77 | 98.00 | 768.3 | 4,962 |
| Mean predator length | cm | 128,395 | 4,252 | 29.00 [20.00, 45.00] | 34.55 ± 22.06 | 98.00 | 768.3 | 4,750 |
| Stomachs represented | count | 132,647 | 0 | 1.00 [1.00, 1.00] | 3.63 ± 30.25 | 55.00 | 3,207.0 | 8,880 |
| Empty stomachs | count | 132,548 | 99 | 0.00 [0.00, 1.00] | 0.85 ± 13.26 | 8.00 | 2,735.0 | 2,326 |
| Minimum prey number | count | 283,117 | 4 | 1.00 [1.00, 1.00] | 87.71 ± 3,177.5 | 319.8 | 574,910 | 65,733 |
| Calculated prey weight | g | 283,117 | 4 | 0.36 [0.00018, 1.96] | 24.94 ± 2,594.9 | 199.4 | 924,486 | 46,674 |

## 7. Missingness and Data-Quality Checks

Highest missingness rates among inspected analytical fields:

| Table | Field | Rows | Null | Blank text | Missing rate |
| --- | --- | --- | --- | --- | --- |
| PREY 6-4 COMBINED | prey_length | 283,121 | 257,290 | 9,382 | 94.2% |
| PREDATOR 6-4 COMBINED | pred_mat | 132,647 | 92,518 | 0 | 69.7% |
| PREDATOR 6-4 COMBINED | pred_wgt_g | 132,647 | 92,489 | 0 | 69.7% |
| PREY 6-4 COMBINED | ind_prey_wgt_g | 283,121 | 192,465 | 2,726 | 68.9% |
| HAUL 6-4 COMBINED | haul_time | 10,262 | 741 | 5,874 | 64.5% |
| HAUL 6-4 COMBINED | shot_time | 10,262 | 604 | 4,653 | 51.2% |
| HAUL 6-4 COMBINED | shot_depth_m | 10,262 | 1,366 | 3,388 | 46.3% |
| MURRAY_TAXONOMY | species | 1,805 | 622 | 0 | 34.5% |
| HAUL 6-4 COMBINED | shot_lon_dd | 10,262 | 3,463 | 0 | 33.7% |
| HAUL 6-4 COMBINED | shot_lat_dd | 10,262 | 3,454 | 0 | 33.7% |
| MURRAY_TAXONOMY | adult_functional_grp | 1,805 | 417 | 0 | 23.1% |
| HAUL 6-4 COMBINED | ices_rect | 10,262 | 1 | 1,563 | 15.2% |
| HAUL 6-4 COMBINED | date | 10,262 | 817 | 258 | 10.5% |
| MURRAY_TAXONOMY | aphiaid | 1,805 | 149 | 0 | 8.3% |
| PREDATOR 6-4 COMBINED | pred_length_cm | 132,647 | 10,004 | 0 | 7.5% |
| HAUL 6-4 COMBINED | Day | 10,262 | 574 | 0 | 5.6% |
| HAUL 6-4 COMBINED | Month | 10,262 | 510 | 0 | 5.0% |
| PREDATOR 6-4 COMBINED | mean_length_cm | 132,647 | 4,252 | 0 | 3.2% |
| PREDATOR 6-4 COMBINED | tpl | 132,647 | 13 | 4,012 | 3.0% |
| PREDATOR 6-4 COMBINED | num_empty | 132,647 | 99 | 0 | 0.1% |

Integrity and domain checks:

| Check | Flagged rows | Status | Interpretation/action |
| --- | --- | --- | --- |
| duplicate_haul_id_rows | 0 | PASS | HAUL primary-key duplicates |
| duplicate_pred_id_rows | 0 | PASS | PREDATOR primary-key duplicates |
| duplicate_prey_id_rows | 0 | PASS | PREY primary-key duplicates |
| predator_orphan_haul_id | 0 | PASS | PREDATOR rows without a matching HAUL |
| prey_orphan_pred_id | 0 | PASS | PREY rows without a matching PREDATOR |
| haul_without_ices_row | 1 | REVIEW | HAUL rows missing from the auxiliary ICES table |
| num_empty_gt_num_stomachs | 27 | REVIEW | `num_empty` greater than represented stomachs; inspect source |
| negative_min_num | 0 | PASS | Negative minimum prey quantities |
| negative_cpw | 0 | PASS | Negative calculated prey weights |
| fractional_min_num | 2 | REVIEW | Fractional `min_num`; retain as recorded pending interpretation |
| fractional_num_empty | 5 | REVIEW | Fractional `num_empty`; retain as recorded pending interpretation |
| latitude_out_of_range | 0 | PASS | Latitude outside [-90, 90] |
| longitude_out_of_range | 0 | PASS | Longitude outside [-180, 180] |

Rows marked `REVIEW` remain in the source-derived summaries until their semantics are checked against provenance; they are not silently corrected or deleted.

## 8. Known Limitations and Required Controls

| Issue | Evidence | Risk | Control / mitigation |
| --- | --- | --- | --- |
| Uneven effort | Sampling varies strongly by sea, decade and source. | Raw network size can track effort rather than ecology. | Report hauls/stomachs with every network; use thresholds, offsets or rarefaction. |
| Pooled stomachs | 9,879 pooled rows represent 74.5% of stomachs. | A database row is not a uniform sampling unit. | Weight/model by represented stomachs and run pooled-vs-individual sensitivity analyses. |
| Observation is not absence | Unrecorded pairs can be unsampled or unresolved. | Naive zeros and random negative sampling create trivial predictions. | Construct pseudo-absences within comparable strata and evaluate grouped holdouts. |
| Taxonomic fragmentation | 5,482 prey labels map to 1,718 prey TSN. | Name-based graphs inflate node and edge counts. | Use TSN as node ID; preserve raw labels for traceability. |
| Non-positive prey TSN | 53,061 rows (18.7%) are excluded from the resolved-taxon primary graph. | Empty/non-trophic and unresolved trophic evidence have different meanings. | Separate empty/non-trophic codes; retain -99913 to -99915 in an unresolved-evidence sensitivity analysis. |
| Calculated weights | `cpw` is calculated and method-dependent. | It is not equivalent to directly observed biomass. | Compare occurrence, `min_num` and `cpw` weightings. |
| Text-encoded numerics | Depth and several length/weight fields are VARCHAR. | Silent coercion can create missing or invalid values. | Parse with explicit unit/domain rules and retain raw columns. |
| Version lineage | The local file/schema is labelled 6.4 while the linked Cefas public description/report refers to 6.3. | A publication could cite the wrong release. | Identify the exact local snapshot by SHA-256 and confirm 6.4 release lineage with Cefas. |

## 9. Network Definition and Global Properties

Primary resolved network:

- Node identity: unified positive TSN across predator and prey roles.
- Edge: `predator_tsn → prey_tsn`, aggregated within `sea × decade`.
- Self-links: retained as potential cannibalism, counted separately, removed from the denominator of the primary density `L_nonself/[S(S−1)]`.
- Sensitivity connectance: `L/S²`, allowing self-link opportunities.
- Edge attributes: prey-record frequency, distinct-haul support, summed `min_num` and summed `cpw`.

| Property | Value | Definition / caveat |
| --- | --- | --- |
| Raw prey records | 283,121 | All PREY rows linked to predators |
| Resolved-taxon prey records | 230,060 | `prey_tsn > 0` |
| Excluded non-positive-TSN rows | 53,061 | Kept separately for sampling/QC and sensitivity analyses |
| Raw name-pair links | 17,824 | Predator/prey labels; vulnerable to synonym fragmentation |
| Resolved name-pair links | 17,043 | Positive prey TSN but still grouped by labels |
| Unified TSN nodes (S) | 1,787 | Union of predator and prey TSN across primary networks |
| Unified directed TSN links (L) | 10,163 | Predator TSN → consumed prey TSN |
| Self-links | 31 | Potential cannibalism; retained and reported separately |
| Primary non-self density | 0.0032 | (L - self-links) / [S(S - 1)] |
| Sensitivity connectance | 0.0032 | L / S², including self-link opportunities |

## 10. Sea × Decade Network Metrics

Metrics were generated for **135** resolved-taxon sea-decade networks. These values describe complete stratum graphs, not only the readable backbones shown in the visual notebook.

| Sea | Decade | Hauls | S | L | Non-self density | L/S | Weak components | Largest component |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| North Sea | 1900 | 571 | 523 | 1,661 | 0.0061 | 3.18 | 2 | 99.6% |
| North Sea | 1970 | 294 | 391 | 1,630 | 0.0106 | 4.17 | 1 | 100.0% |
| Irish Sea | 1980 | 146 | 169 | 854 | 0.0300 | 5.05 | 1 | 100.0% |
| Celtic Sea | 1990 | 225 | 178 | 845 | 0.0267 | 4.75 | 1 | 100.0% |
| North Sea | 1990 | 355 | 314 | 837 | 0.0085 | 2.67 | 1 | 100.0% |
| Channel | 1910 | 492 | 204 | 747 | 0.0180 | 3.66 | 1 | 100.0% |
| North Sea | 2000 | 518 | 247 | 741 | 0.0122 | 3.00 | 1 | 100.0% |
| North Sea | 1880 | 371 | 271 | 740 | 0.0101 | 2.73 | 1 | 100.0% |
| North Sea | 1980 | 187 | 271 | 547 | 0.0074 | 2.02 | 1 | 100.0% |
| North Sea | 1890 | 194 | 167 | 527 | 0.0189 | 3.16 | 1 | 100.0% |
| Celtic Sea | 2010 | 110 | 226 | 520 | 0.0102 | 2.30 | 1 | 100.0% |
| Irish Sea | 1950 | 49 | 183 | 516 | 0.0155 | 2.82 | 1 | 100.0% |

## 11. Representative Individual Networks and Structural Differences

Selection is deterministic: among strata with at least 30 hauls, 10 predator taxa and 100 prey taxa, choose high, median and low directed-edge complexity while preferring different seas. Metrics refer to each **complete** graph; visual figures show a labelled high-support backbone for readability.

| Complexity tier | Sea | Decade | Hauls | S | L | Density | L/S | Top-10 record share |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| High | North Sea | 1900 | 571 | 523 | 1,661 | 0.0061 | 3.18 | 13.5% |
| Median | Channel | 1930 | 46 | 151 | 443 | 0.0196 | 2.93 | 13.9% |
| Low | Irish Sea | 1900 | 50 | 134 | 197 | 0.0111 | 1.47 | 40.0% |

Among these examples, **Channel 1930** has the highest non-self density (0.0196), while **Irish Sea 1900** has the greatest concentration of prey records in its ten strongest links (40.0%). These contrasts are descriptive: graph structure and sampling effort/provenance remain entangled.

## 12. Main Research Questions

| ID | Research question |
| --- | --- |
| RQ1 | How do resolved food-web size, density, degree structure and components vary among seas and decades after controlling for effort? |
| RQ2 | How robust are inferred structures to spatial grain, temporal binning, pooled records and edge weighting (`occurrence`, `min_num`, `cpw`)? |
| RQ3 | Which predator traits, prey taxonomy and environmental/context variables explain observed interactions and diet breadth? |
| RQ4 | Can link-prediction models recover held-out interactions in new campaigns, decades, seas or predators better than simple baselines? |
| RQ5 | How much do provenance, taxonomic resolution and non-positive/unresolved prey codes change ecological conclusions? |

## 13. Required Preprocessing

| Order | Operational rule |
| --- | --- |
| 1. Freeze and trace | Record source path, checksum, extraction date, schema and software versions; never overwrite the Access source. |
| 2. Validate keys | Enforce HAUL → PREDATOR → PREY and PROVENANCE links; review auxiliary ICES gaps and duplicated IDs. |
| 3. Parse fields | Convert text-encoded depth/length/weight with explicit missing tokens, units and domain checks; retain raw values. |
| 4. Normalise taxonomy | Use TSN as primary node ID, attach canonical name/AphiaID, preserve qualifiers, and log synonym merges. |
| 5. Classify evidence | Separate empty/non-trophic codes, resolved positive TSN, and unresolved trophic codes (-99913 to -99915). |
| 6. Define sampling unit | Keep haul, campaign, pooling and represented-stomach fields; do not treat every predator row as equal effort. |
| 7. Construct edges | Aggregate within declared sea × decade (then test alternative grains); retain occurrence, `min_num`, `cpw` and distinct-haul support. |
| 8. Apply eligibility | Set minimum effort/taxon support before comparisons; record excluded strata and run threshold sensitivity analyses. |
| 9. Build evaluation splits | Hold out campaigns/time/space/predators as groups and sample negatives only within ecologically comparable candidate sets. |

## 14. Candidate Network and Statistical Analyses

| Analysis family | Candidate analyses |
| --- | --- |
| Network description | S, L, self-links, non-self density, L/S, in/out/total degree, weighted strength, components, largest-component share, basal/intermediate/top fractions. |
| Structural comparison | Effort-aware comparisons across sea/decade; rarefaction or coverage standardisation; sensitivity to graph grain and edge weights. |
| Community/trophic structure | Modularity, motifs, trophic position/height and generality/vulnerability where taxonomic resolution supports them. |
| Composition | Ordination and permutation tests for diet/link composition, with campaign/sea/decade restrictions and effort-aware distances. |
| Statistical models | GLMM/GAM or hierarchical models for edge presence/weight and network metrics; random effects for campaign/predator and nonlinear time trends. |
| Link prediction | Degree/common-neighbour heuristics, trait baselines, WLNM/SEAL variants and calibrated classifiers evaluated on grouped holdouts. |
| Sensitivity/null models | Pooling, unresolved prey, taxonomic aggregation, effort thresholds, weighting schemes and degree-preserving/ecological null models. |

## 15. Assumptions, Risks and Mitigations

| Assumption / risk | Potential impact | Mitigation |
| --- | --- | --- |
| Observed links represent realised diet; non-links are not confirmed absences. | False negatives and inflated model performance. | Use constrained pseudo-absences and grouped external-style holdouts. |
| Sampling effort and source protocols are exchangeable after adjustment. | Residual confounding across time/space/source. | Include effort/provenance covariates and stratified sensitivity analyses. |
| TSN resolves biological identity consistently through time. | Synonyms, aggregated prey and population labels distort nodes. | Audit mappings; publish raw-to-canonical crosswalk and resolution flags. |
| `min_num` and `cpw` are comparable across source formats. | Weight-based trends may reflect derivation methods. | Analyse occurrence, counts and calculated weight separately. |
| Sea × decade is an ecologically meaningful first grain. | Aggregation can hide seasonality and local structure. | Repeat at division/rectangle/season where sample size permits. |
| Random ML splits are representative. | Leakage through campaign, predator or repeated edges. | Prohibit random-edge-only claims; use grouped and temporal/spatial transfer tests. |

## 16. Expected Outputs and Acceptance Checks

| Output | Required contents | Validation / acceptance |
| --- | --- | --- |
| Versioned extraction manifest | Source checksum, schema, query log and generated-file inventory. | Checksum match; zero extractor errors; deterministic rerun. |
| Clean relational tables and crosswalks | Typed fields, retained raw values, key/taxonomy/provenance mappings. | Unique PKs, resolved FKs, documented parsing failures and reconciliation totals. |
| Edge tables by declared stratum | TSN-to-TSN directed links with occurrence, haul support, `min_num`, `cpw`, effort and provenance. | No non-positive prey TSN in primary graph; raw = retained + excluded; duplicate aggregation checked. |
| Network/node metrics | Graph-level and node-level metrics with definitions and loop policy. | Metric bounds; sum in-degree = sum out-degree = L; sensitivity definitions agree. |
| EDA figures | Coverage, distributions, missingness, effort and representative individual networks. | Selection/layout deterministic; legends, units and filters visible. |
| Statistical/model results | Effect estimates, uncertainty, predictions, grouped splits and baselines. | No group leakage; calibration/discrimination reported; outperform declared baselines. |
| Sensitivity and QC report | Pooling, taxonomy, weighting, grain, threshold and unresolved-evidence analyses. | Conclusions labelled robust or conditional; all review flags resolved or justified. |

## 17. Current Automated Validation

| Check | Status | Observed | Expected | Meaning |
| --- | --- | --- | --- | --- |
| primary_network_positive_prey_tsn | PASS | 0 | 0 | Primary sea-decade networks must exclude non-positive prey TSN codes. |
| network_metric_bounds | PASS | 0 | 0 | Every graph must satisfy L <= S^2 and both reported density measures must lie in [0, 1]. |
| degree_edge_conservation | PASS | 0 | 0 | For each directed network, sum(in-degree) = sum(out-degree) = L. |
| prey_record_filter_accounting | PASS | 283121 = 230060 + 53061 | raw = retained + excluded | The non-positive-TSN filter must reconcile exactly to the raw prey-record total. |
| representative_network_count | PASS | 3 | 3 | The deterministic high/median/low selection should produce three eligible networks. |
| extractor_query_errors | PASS | 0 | 0 | All Access extraction queries must complete without recorded errors. |
| key_uniqueness_and_referential_integrity | PASS | 0 | 0 | Primary identifiers must be unique and HAUL -> PREDATOR -> PREY foreign keys must resolve. |
| auxiliary_ices_table_coverage | REVIEW | 1 | 0 | The auxiliary ICES table should be reconciled before ICES-specific analyses. |
| semantic_count_anomalies | REVIEW | 34 | 0 | Flagged rows require source-level review; they are not deleted automatically. |
| source_and_working_copy_match | PASS | 5a1346cd28e98b73ec2a51819408ccf333b76b108a39d2c3f3d3a3554a6f290e | identical byte content | The original supplied Access file and the read-only working copy must be byte-identical. |

`REVIEW` is not an extraction failure: it identifies source-level semantic anomalies requiring a documented decision before modelling.

## 18. Realistic Implementation Sequence

| Phase | Work | Completion gate |
| --- | --- | --- |
| Phase 1 — Provenance and dictionary | Freeze snapshot, citation/version note, schema and units. | Manifest, checksum and compact data dictionary complete. |
| Phase 2 — QC and preprocessing | Keys, missingness, parseability, semantic anomalies, taxonomy and evidence classes. | QC report passes or every exception has a recorded decision. |
| Phase 3 — Graph construction | Create reproducible primary and sensitivity edge lists by stratum. | Counts reconcile and graph validation identities pass. |
| Phase 4 — EDA and network metrics | Describe effort, distributions, topology and representative graphs. | All figures/tables regenerate and comparisons include effort. |
| Phase 5 — Statistical and link-prediction models | Fit baselines and candidate models with grouped splits. | Pre-registered metrics, uncertainty and leakage checks complete. |
| Phase 6 — Sensitivity and synthesis | Repeat across pooling, weights, grain, taxonomy and thresholds. | Final claims distinguish robust findings from assumption-dependent findings. |

## 19. Reproducibility and Generated Files

From the repository root:

```bash
bash data/processed/dapstom_eda/tools/run_extractor.sh
python3 data/processed/dapstom_eda/tools/derive_network_metrics.py
python3 data/processed/dapstom_eda/tools/build_notebook.py
python3 data/processed/dapstom_eda/tools/build_visual_notebook.py
```

Key generated artifacts:

- `tables/eda_manifest.json`: timestamp, source path, size, checksum and extraction driver.
- `tables/numeric_variable_summary.csv` and `numeric_variable_histograms.csv`: continuous-variable EDA.
- `tables/data_quality_checks.csv` and `validation_checks.csv`: QC and invariant checks.
- `tables/sea_decade_network_edges_positive_prey_tsn.csv`: filtered stratum edge evidence.
- `tables/sea_decade_network_metrics.csv` and `sea_decade_node_degrees.csv`: graph/node metrics.
- `tables/representative_networks.csv`: deterministic graph selection.
- `figures/*.svg`: reproducible visual EDA and representative networks.

The saved Access view `Query1` emits a known non-blocking duplicate-column warning; extraction uses base tables with explicit aliases, and `query_errors.csv` must remain empty.